# cvfoodid -- Train YOLOv8 ingredient detector on Colab

End-to-end walkthrough: clone repo -> download datasets -> convert to YOLO format -> train -> evaluate -> export TFLite (INT8) for mobile.

**Recommended runtime:** *Runtime -> Change runtime type -> T4 GPU* (free) or *L4* / *A100* (Pro+).

**Time:** ~2-3 hours for 100 epochs of YOLOv8n on FoodSeg103 + Padang Cuisine on a T4.

## 1. Clone the repo and install

In [ ]:
# If you forked the repo, change the URL below.
REPO_URL = 'https://github.com/OyenCar/cv-food-id2.git'
REPO_DIR = REPO_URL.split('/')[-1].replace('.git', '')

!git clone $REPO_URL
%cd $REPO_DIR
!pip install --quiet -e ".[ml,demo]"
!pip install --quiet kaggle

## 2. Verify GPU and Ultralytics import

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
from ultralytics import YOLO
print('Ultralytics OK')

## 3. Authenticate Kaggle (for Padang & BEEI subsets)

Upload your `kaggle.json` from https://www.kaggle.com/settings/account:

In [ ]:
from google.colab import files
import os
uploaded = files.upload()
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

## 4. Download datasets

FoodSeg103 (Apache 2.0, 9.5k images, 103 ingredient masks) is the workhorse. Padang Cuisine and Indonesian Mendeley add Indonesian dish coverage.

In [ ]:
!python scripts/download_datasets.py foodseg103 padang-cuisine indonesian-traditional-foods
!ls -lh data/raw

FoodSeg103 ships password-protected. If extraction fails:

```
!unzip -P LARCdataset9947 data/raw/foodseg103/FoodSeg103.zip -d data/raw/foodseg103
```

## 5. Convert masks to YOLO bbox labels

In [ ]:
!python scripts/prepare_yolo_dataset.py \
    --raw data/raw/foodseg103/FoodSeg103 \
    --out data/processed/yolo \
    --data-yaml configs/data.yaml
!ls data/processed/yolo/images/train | head
!ls data/processed/yolo/labels/train | head

## 6. Train YOLOv8n

Defaults: 100 epochs, 640 imgsz, batch 16. Adjust to your runtime.

In [ ]:
!python scripts/train_yolo.py \
    --data configs/data.yaml \
    --model yolov8n.pt \
    --epochs 100 \
    --imgsz 640 \
    --batch 16 \
    --name cvfoodid-yolo

## 7. Evaluate

In [ ]:
from ultralytics import YOLO
model = YOLO('runs/detect/cvfoodid-yolo/weights/best.pt')
metrics = model.val(data='configs/data.yaml', imgsz=640)
print('mAP50:', metrics.box.map50, 'mAP50-95:', metrics.box.map)

## 8. Export to TFLite (INT8) for mobile

In [ ]:
!python scripts/export_tflite.py \
    --weights runs/detect/cvfoodid-yolo/weights/best.pt \
    --imgsz 640 \
    --calib data/processed/yolo/images/val \
    --int8
!ls -lh runs/detect/cvfoodid-yolo/weights/

## 9. Inference + nutrition computation

Continue in `02_inference_demo.ipynb`.